# Set up

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from collections import Counter
from matplotlib import cm
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, explained_variance_score
import random
import re
import os
from pathlib import Path
import time
import gc
from tqdm.auto import tqdm

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

# Configuration

In [ ]:
base_dir = Path('/scratch/bng/cartbind/code/MIND_models')
coefs_dir = base_dir / 'models_plsregression_dnanexus/PLS_coefs_scaling_law'
data_dir = Path('/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers')
splits_dir = base_dir / 'scaling_law_splits'
region_dir = base_dir / 'region_names'

weights_dir = base_dir / 'models_plsregression_dnanexus/PLS_weights_scaling_law'
results_dir = base_dir / 'models_plsregression_dnanexus/PLS_scaling_law_results'
predictions_dir = base_dir / 'models_plsregression_dnanexus/PLS_predictions_scaling_law'
os.makedirs(weights_dir, exist_ok=True)
os.makedirs(coefs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
os.makedirs(predictions_dir, exist_ok=True)

rename = pd.read_csv(region_dir / 'col_renames_dnanexus.csv')
rename_dict = dict(zip(rename['datafield_code'], rename['datafield_name']))

targets = {
    'GF': ('GF', 'p20016_i2'),
    'PAL': ('PAL', 'p20197_i2'),
    'DSST': ('DSST', 'p23324_i2'),
    'TMT': ('TMT', 'p6350_i2'),
}

data_configs = {
    'demo': (None, ['p31', 'p21003_i2', 'p54_i2']),
    'MIND_avg': (region_dir / 'MIND_avg_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),
    'CT': (region_dir / 'CT_regions_dnanexus.txt', ['p31', 'p21003_i2', 'p54_i2']),
    'FC25': (region_dir / 'FC25_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'FC100': (region_dir / 'FC100_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'MIND': (region_dir / 'MIND_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),
}

sample_sizes = [250, 500, 1000, 2000, 4000, 8000, 16000, 32000, 'all']

# PLS Analysis Function

In [ ]:
# inner parallelized
def pls_analysis(X, y, continuous_vars, categorical_vars, weights_dir, coefs_dir, predictions_dir, data_name, target_name, sample_size, n_splits=10):
    preprocessor = ColumnTransformer(transformers=[
        # scale continuous features
        ('num', StandardScaler(), continuous_vars),
        # one-hot encode assessment centre and sex
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_vars),
    ])

    # Demo-only has very few features; cap components accordingly
    max_n_components = 5 if data_name == 'demo' else 10

    # Cross-validation set-up
    outer_cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    inner_cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed+1)
    
    outer_mae, outer_rmse, outer_r2, outer_r2_corr = [], [], [], []

    target_weights_dir = os.path.join(weights_dir, target_name)
    os.makedirs(target_weights_dir, exist_ok=True)
    target_coefs_dir = os.path.join(coefs_dir, target_name)
    os.makedirs(target_coefs_dir, exist_ok=True)
    target_predictions_dir = os.path.join(predictions_dir, target_name)
    os.makedirs(target_predictions_dir, exist_ok=True)
    preds_filename = f'PLS_preds_{data_name}_{target_name}_{sample_size}.csv'
    preds_path = os.path.join(target_predictions_dir, preds_filename)

    # 'regressor__' prefix b/c pipeline is wrapped in TransformedTargetRegressor
    param_grid = {
        'regressor__plsregression__n_components': list(range(1, max_n_components + 1))
    }

    # Inner model
    inner_model = TransformedTargetRegressor(
        regressor=make_pipeline(
            preprocessor,
            PLSRegression(scale=False)
        ),
        transformer=StandardScaler()
    )

    cv_splits = tqdm(
        outer_cv.split(X, y), 
        total=n_splits, 
        desc=f"CV Folds ({target_name} on {data_name}, n={sample_size})", 
        leave=False
    )
    
    for fold, (train_idx, test_idx) in enumerate(cv_splits, start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Inner CV Grid Search
        grid = GridSearchCV(
            inner_model, 
            param_grid, 
            cv=inner_cv, 
            scoring='neg_mean_squared_error',
            n_jobs=-1
        )
        
        grid.fit(X_train, y_train.values.reshape(-1, 1))
        
        best_n = grid.best_params_['regressor__plsregression__n_components']
        
        # Predictions are automatically unscaled
        y_pred = grid.predict(X_test).ravel()

        fold_df = pd.DataFrame({
            'fold': fold,
            'eid': y_test.index,
            'actual': y_test.values,
            'predicted': y_pred
        })
        fold_df.to_csv(preds_path, mode='a', header=(fold == 1), index=False)
        del fold_df, grid
        gc.collect()

        # metrics
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        if np.std(y_pred) == 0:
             corr = 0.0
        else:
             corr = np.corrcoef(y_test, y_pred.squeeze())[0, 1]
        r2_corr = corr ** 2

        outer_mae.append(mae)
        outer_rmse.append(rmse)
        outer_r2.append(r2)
        outer_r2_corr.append(r2_corr)

        print(f'  Fold {fold:02d} • n_comp={best_n:02d} • MAE={mae:.3f} • RMSE={rmse:.3f} • R²={r2:.3f} • R²(corr)={r2_corr:.3f}')

    print(f'\n  Mean MAE    : {np.mean(outer_mae):.3f} ± {np.std(outer_mae):.3f}')
    print(f'  Mean RMSE   : {np.mean(outer_rmse):.3f} ± {np.std(outer_rmse):.3f}')
    print(f'  Mean R²     : {np.mean(outer_r2):.3f} ± {np.std(outer_r2):.3f}')
    print(f'  Mean R²(corr): {np.mean(outer_r2_corr):.3f} ± {np.std(outer_r2_corr):.3f}')

    # Final refit on all data
    final_grid = GridSearchCV(
        inner_model, 
        param_grid, 
        cv=inner_cv, 
        scoring='neg_mean_squared_error',
        n_jobs=-1
    ).fit(X, y.values.reshape(-1, 1))

    final_ncomps = final_grid.best_params_['regressor__plsregression__n_components']

    # Unpack the best model components to extract weights and coefficients
    pls = final_grid.best_estimator_.regressor_.named_steps['plsregression']
    W = pls.x_weights_
    coefficients = pls.coef_.squeeze()

    # Feature names after preprocessing
    preprocessor_fitted = final_grid.best_estimator_.regressor_.named_steps['columntransformer']
    cat_features = list(preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(categorical_vars))
    all_feature_names = continuous_vars + cat_features

    # Save the PLS x_weights_
    weights_df = pd.DataFrame(
        W,
        index=all_feature_names,
        columns=[f'Component_{i+1}' for i in range(W.shape[1])]
    )
    weights_filename = f'PLS_weights_{data_name}_{target_name}_{sample_size}.csv'
    weights_df.to_csv(os.path.join(target_weights_dir, weights_filename), index_label='Feature')

    # Save the regression coefficients
    coefs_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Coefficient': coefficients
    })
    coefs_filename = f'PLS_coefs_{data_name}_{target_name}_{sample_size}.csv'
    coefs_df.to_csv(os.path.join(target_coefs_dir, coefs_filename), index=False)
    
    del weights_df, coefs_df, final_grid, pls
    gc.collect()

    print(f'  Weights → {weights_filename}')
    print(f'  Coefs   → {coefs_filename}')
    print(f'  Predictions → {preds_filename}')
    print(f'  Final params: n_components={final_ncomps}')
    
    return {
        'mean_mae':       np.mean(outer_mae),
        'std_mae':        np.std(outer_mae),
        'mean_rmse':      np.mean(outer_rmse),
        'std_rmse':       np.std(outer_rmse),
        'mean_r2':        np.mean(outer_r2),
        'std_r2':         np.std(outer_r2),
        'mean_r2_corr':   np.mean(outer_r2_corr),
        'std_r2_corr':    np.std(outer_r2_corr),
        'n_components':   final_ncomps,
    }

# Scaling law training loop

In [ ]:
for target_name, (test_key, score_col) in targets.items():
    print(f'\n{"="*60}\nTARGET: {target_name}\n{"="*60}')

    data_file = data_dir / f'combined_data_{test_key}_no_outliers.csv'
    target_splits_dir = splits_dir / target_name

    df_full = pd.read_csv(data_file, index_col=0)

    for data_name, (regions_file, demographic_vars) in data_configs.items():
        print(f'\n--- {target_name} vs. {data_name} ---')

        if regions_file is not None:
            with open(regions_file, 'r') as f:
                brain_regions = [line.strip() for line in f]
        else:
            brain_regions = []

        all_vars = demographic_vars + brain_regions

        for sample_size in sample_sizes:
            if sample_size == 'all':
                eid_file = target_splits_dir / f'{target_name}_all_eids.txt'
            else:
                eid_file = target_splits_dir / f'{target_name}_eids_{sample_size}.txt'

            if not eid_file.exists():
                print(f'  Skipping n={sample_size}: EID file not found.')
                continue

            sample_eids = np.loadtxt(eid_file, dtype=int)

            if 'eid' in df_full.columns:
                df = df_full[df_full['eid'].isin(sample_eids)]
            else:
                df = df_full[df_full.index.isin(sample_eids)]

            actual_n = len(df)
            print(f'\n[{target_name} | {data_name} | n={sample_size} ({actual_n} rows)]')

            X = df[all_vars].rename(columns=rename_dict)
            y = df[score_col]

            categorical_vars = ['sex', 'assessment_centre']
            continuous_vars  = [c for c in X.columns if c not in categorical_vars]

            start_time = time.time()

            metrics = pls_analysis(
                X, y, continuous_vars, categorical_vars,
                weights_dir, coefs_dir, predictions_dir, data_name, 
                target_name, sample_size
            )

            end_time = time.time()  # End timer
            elapsed = end_time - start_time
            print(f'  Time taken for config [{data_name}, n={sample_size}]: {elapsed:.2f} seconds')

            results_file = results_dir / f'scaling_law_results_{target_name}.csv'
            row_df = pd.DataFrame([{
                'target_name': target_name,
                'data_name':   data_name,
                'sample_size': sample_size,
                'actual_n':    actual_n,
                **metrics,
                'elapsed_time_sec': elapsed
            }])
            row_df.to_csv(str(results_file), mode='a', header=not results_file.exists(), index=False)

            # Explicit memory cleanup
            del df, X, y, metrics
            gc.collect()

    print(f'\nResults saved → {results_file}')